# TP53 Mutation Subtype -- Multi-Window Data Pipeline

Builds sequence-context datasets for TP53 single-nucleotide substitution
classification at four context window sizes: **11bp, 21bp, 51bp, 101bp**,
each centered on the mutated nucleotide.

**Pipeline stages**

1. Load raw per-transcript mutation tables (`data/raw/csvFolder/gene_*.csv`) and
   matching FASTA sequences (`data/raw/fastaFolder/file_*.fasta`).
2. For each transcript and each window size, extract substitution mutations,
   validate the reference base, and cut out the centered sequence window.
   Every instance row is tagged with a stable `position_id` (`gene_number_cdsPos`)
   that identifies the transcript-relative locus independent of window size.
3. Normalize mutation type to the six pyrimidine-based substitution classes
   (C>A, C>G, C>T, T>A, T>C, T>G).
4. Decide which positions are valid using **only the 101bp window** (the
   largest): a position survives if its 101bp context contains no boundary
   `N` padding. This single fixed set of valid `position_id`s is then applied
   to every window size, so all four window-size datasets describe the exact
   same set of genomic positions.
5. **Cluster `position_id`s into locus clusters** by exact sequence identity
   at the *smallest* window size (11bp). TP53's 19 `gene_N`/`file_N` inputs
   are not 19 different genes -- they are 19 transcript isoforms, several of
   which share long identical exonic stretches, and COSMIC re-annotates the
   same genomic variant once per affected transcript. Left alone,
   `position_id` (transcript, cds_pos) treats each of those redundant
   annotations as a distinct position, which both (a) fragments/inflates
   `n_instances` for what is really one locus observed via multiple
   transcripts, and (b) leaks identical sequence content across train/val/test
   when the split is done on raw `position_id`s. Clustering on the smallest
   window is the one choice that closes this for all four window sizes at
   once: because every window is a substring of every larger window centered
   on the same base, two positions identical at 11bp are the necessary
   condition for identical content at any larger window, and identical
   content at any larger window always implies identical 11bp content.
6. Build one canonical **cluster-level** table (one row per locus cluster)
   with instance counts, majority subtype, subtype diversity, and a hotspot
   flag (`n_instances >= 100`), aggregated across every `position_id` in the
   cluster -- this is the correct resolution for "how mutated is this locus",
   not the raw per-transcript-annotation counts.
7. Run a stratified, cluster-based 70:15:15 train/val/test split **exactly
   twice** -- once on hotspot clusters, once on rare-variant clusters --
   stratified by majority subtype. This produces two `cluster_id -> split`
   assignment tables, independent of window size.
8. For each window size, build train/val/test CSVs by joining instance rows
   (via `position_id -> cluster_id`) to the cluster-level split assignment
   (never re-split per window size, never split on raw `position_id`).

This guarantees identical train/val/test position sets across all four
window sizes (required for a fair window-size ablation), separates the
hotspot and rare-variant regimes into two independent experiments, and --
new in this revision -- guarantees no exact-duplicate sequence content
crosses a train/val/test boundary, at any window size.

Outputs land under `data/processed/` and `data/splits/`.


In [1]:
import os
import re
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

RAW_CSV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "csvFolder")
RAW_FASTA_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "fastaFolder")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SPLITS_DIR = os.path.join(PROJECT_ROOT, "data", "splits")

WINDOW_SIZES = [11, 21, 51, 101]
FILTER_WINDOW = max(WINDOW_SIZES)   # N-validity is decided once, using the largest window
CLUSTER_WINDOW = min(WINDOW_SIZES)  # locus clustering is decided once, using the smallest window

HOTSPOT_THRESHOLD = 100  # n_instances >= this -> hotspot cluster

TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Raw mutation csvs:  {RAW_CSV_DIR}")
print(f"Raw fasta files:    {RAW_FASTA_DIR}")
print(f"Window sizes:       {WINDOW_SIZES}")
print(f"N-filter window:    {FILTER_WINDOW}bp (largest)")
print(f"Cluster window:     {CLUSTER_WINDOW}bp (smallest)")
print(f"Hotspot threshold:  >= {HOTSPOT_THRESHOLD} instances")


Project root:      C:\Users\danya\Documents\projects\tp53_mutation_subtype
Raw mutation csvs:  C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\raw\csvFolder
Raw fasta files:    C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\raw\fastaFolder
Window sizes:       [11, 21, 51, 101]
N-filter window:    101bp (largest)
Cluster window:     11bp (smallest)
Hotspot threshold:  >= 100 instances


## Helper functions

In [2]:
def extract_number_from_filename(filename):
    """gene_7.csv / file_7.fasta -> 7, so csv/fasta pairs line up."""
    return int(filename.split('_')[1].split('.')[0])


def read_fasta_sequence(fasta_path):
    """Read a single-record FASTA file and return the sequence as a string."""
    seq_lines = []
    with open(fasta_path, 'r') as f:
        for line in f:
            if line.startswith('>'):
                continue
            seq_lines.append(line.strip())
    return ''.join(seq_lines).upper()


def get_kmer(sequence, pos, window_size):
    """Extract a window_size-nt window centered on 1-indexed position pos.

    Pads with 'N' when the window would run off either end of the sequence.
    """
    assert window_size % 2 == 1, "window_size must be odd to center on one base"
    half = window_size // 2
    center_idx = pos - 1  # 0-indexed

    start = center_idx - half
    end = center_idx + half + 1

    left_pad = max(0, -start)
    right_pad = max(0, end - len(sequence))

    kmer = sequence[max(0, start):min(len(sequence), end)]
    return ('N' * left_pad) + kmer + ('N' * right_pad)


COMPLEMENT = {'A': 'T', 'G': 'C', 'T': 'A', 'C': 'G'}


def classify_mutation_type(ref, alt):
    """Normalize substitutions to the six pyrimidine-based classes.

    Purine-reference substitutions (A>?, G>?) are expressed on the
    complementary strand so all mutation types are reported as C>x or T>x.
    """
    if ref in ('C', 'T'):
        return f"{ref}>{alt}"
    return f"{COMPLEMENT[ref]}>{COMPLEMENT[alt]}"


def reverse_complement(seq):
    """Reverse-complement a sequence window, treating boundary 'N' padding as itself.

    Used to keep a mutation's sequence window on the same strand convention
    as its pyrimidine-normalized label (see classify_mutation_type): for a
    purine-reference mutation, the label is re-expressed on the complementary
    strand, so the window must be too, or the model is shown a center base
    (A/G) that contradicts the label's implied center base (C/T), and the
    flanking context is read in the wrong direction entirely.
    """
    comp = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    return ''.join(comp[b] for b in reversed(seq))


MUTATION_PATTERN = re.compile(r'c\.(\d+)([ACGT])>([ACGT])')


## Stage 1-3: extract windowed mutations per transcript, tagged with `position_id`

For every `(gene_N.csv, file_N.fasta)` pair and every window size, parse
single-nucleotide substitutions (`c.<pos><ref>><alt>`), skip indels/complex
mutations and any row whose reference base doesn't match the FASTA sequence,
then emit one row per mutation occurrence (repeated by COSMIC `Count`).

Each row carries `position_id = f"{gene_number}_{cds_pos}"` -- a window-size
independent identifier for the transcript-relative locus -- plus
`gene_number` and `cds_pos` as separate columns for QC. `position_id` alone
does not yet guarantee distinct genomic loci (see Stage 5's locus
clustering); it's the first step, and what makes it possible to guarantee
identical positions across window sizes later, since "position" is no longer
defined by the (window-size-dependent, collision-prone) `Sequence` string.


In [3]:
def extract_gene_windows(csv_path, fasta_path, window_sizes):
    """Return {window_size: list[dict(position_id, gene_number, cds_pos,
    Sequence, MutationType)]} for one transcript, plus a stats dict for QC.
    """
    gene_number = extract_number_from_filename(os.path.basename(csv_path))

    df = pd.read_csv(csv_path)
    sequence = read_fasta_sequence(fasta_path)

    rows = {w: [] for w in window_sizes}
    stats = defaultdict(int)

    for _, row in df.iterrows():
        stats['total_rows'] += 1
        raw_mutation = str(row['CDS Mutation'])

        try:
            count = int(row['Count'])
        except (ValueError, TypeError):
            stats['bad_count'] += 1
            continue

        match = MUTATION_PATTERN.match(raw_mutation)
        if not match:
            stats['non_substitution'] += 1  # indels, splice variants, etc.
            continue

        pos, ref, alt = int(match.group(1)), match.group(2), match.group(3)

        if pos <= 0 or pos > len(sequence):
            stats['out_of_range'] += 1
            continue

        if sequence[pos - 1] != ref:
            stats['ref_mismatch'] += 1
            continue

        stats['valid_substitutions'] += 1
        mutation_type = classify_mutation_type(ref, alt)
        position_id = f"{gene_number}_{pos}"

        for w in window_sizes:
            kmer = get_kmer(sequence, pos, w)
            if ref not in ('C', 'T'):
                # Purine reference: label was re-expressed on the complementary
                # strand (classify_mutation_type), so the window must match.
                kmer = reverse_complement(kmer)
            rows[w].extend([{
                'position_id': position_id,
                'gene_number': gene_number,
                'cds_pos': pos,
                'Sequence': kmer,
                'MutationType': mutation_type,
            }] * count)

    return rows, stats


def run_extraction(csv_dir, fasta_dir, window_sizes, output_dir):
    """Process every gene/fasta pair, save per-transcript windowed CSVs,
    and return combined {window_size: DataFrame} plus aggregated QC stats.
    """
    csv_files = sorted(
        (f for f in os.listdir(csv_dir) if f.startswith('gene_') and f.endswith('.csv')),
        key=extract_number_from_filename,
    )
    fasta_files = sorted(
        (f for f in os.listdir(fasta_dir) if f.startswith('file_') and f.endswith('.fasta')),
        key=extract_number_from_filename,
    )

    assert len(csv_files) == len(fasta_files), (
        f"Mismatch: {len(csv_files)} mutation csvs vs {len(fasta_files)} fasta files"
    )

    combined_rows = {w: [] for w in window_sizes}
    total_stats = defaultdict(int)

    for w in window_sizes:
        os.makedirs(os.path.join(output_dir, f"window_{w}"), exist_ok=True)

    for csv_file, fasta_file in zip(csv_files, fasta_files):
        csv_path = os.path.join(csv_dir, csv_file)
        fasta_path = os.path.join(fasta_dir, fasta_file)

        gene_rows, stats = extract_gene_windows(csv_path, fasta_path, window_sizes)
        for k, v in stats.items():
            total_stats[k] += v

        for w in window_sizes:
            if gene_rows[w]:
                gene_df = pd.DataFrame(gene_rows[w])
                gene_df.to_csv(
                    os.path.join(output_dir, f"window_{w}", csv_file), index=False
                )
                combined_rows[w].extend(gene_rows[w])

        print(f"{csv_file}: {stats['valid_substitutions']} valid substitutions "
              f"(skipped {stats['non_substitution']} non-sub, "
              f"{stats['ref_mismatch']} ref-mismatch, "
              f"{stats['out_of_range']} out-of-range)")

    combined = {w: pd.DataFrame(combined_rows[w]) for w in window_sizes}
    return combined, total_stats


raw_combined_datasets, extraction_stats = run_extraction(
    RAW_CSV_DIR, RAW_FASTA_DIR, WINDOW_SIZES, PROCESSED_DIR
)

print("\nTotals across all transcripts:")
for k, v in extraction_stats.items():
    print(f"  {k}: {v:,}")

# Sanity check: every window size must describe the exact same set of raw
# (pre-filter) positions, since one substitution list drives all window sizes.
raw_position_sets = {
    w: set(raw_combined_datasets[w]['position_id']) for w in WINDOW_SIZES
}
reference_positions = raw_position_sets[WINDOW_SIZES[0]]
for w in WINDOW_SIZES[1:]:
    assert raw_position_sets[w] == reference_positions, (
        f"Raw position_id set for window {w} differs from window {WINDOW_SIZES[0]} "
        "before any N-filtering -- extraction is no longer window-size symmetric."
    )
print(f"\nOK: all {len(WINDOW_SIZES)} window sizes yield the same "
      f"{len(reference_positions):,} raw positions before N-filtering.")


gene_1.csv: 1690 valid substitutions (skipped 3257 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_2.csv: 1824 valid substitutions (skipped 3124 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_3.csv: 1884 valid substitutions (skipped 3061 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_4.csv: 1824 valid substitutions (skipped 3125 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_5.csv: 1750 valid substitutions (skipped 3195 non-sub, 0 ref-mismatch, 0 out-of-range)


...

gene_17.csv: 1487 valid substitutions (skipped 2351 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_18.csv: 1152 valid substitutions (skipped 2689 non-sub, 0 ref-mismatch, 0 out-of-range)


gene_19.csv: 1884 valid substitutions (skipped 4236 non-sub, 0 ref-mismatch, 0 out-of-range)



Totals across all transcripts:
  total_rows: 88,218
  non_substitution: 57,568
  valid_substitutions: 30,650



OK: all 4 window sizes yield the same 14,355 raw positions before N-filtering.


## Stage 4: fix the N-filter -- decide validity ONCE using the largest window

Previously, sequences containing boundary `N` padding were dropped
independently per window size, so the surviving position set differed across
window sizes before splitting even happened. That's fixed here: a
`position_id` is retained if and only if its **101bp** window has no `N`.
That fixed set of `valid_position_ids` is then applied identically to every
window size -- an 11bp window is never independently N-checked, even though
in isolation it might have no `N` (its underlying position may still be
too close to a transcript boundary for the 101bp context).

We also report how many positions are lost purely because of this
standardization: positions whose smaller-window sequence has no `N`, but
whose 101bp sequence does.


In [4]:
# has_N flag per position_id, per window size, computed on the *raw* (unfiltered) data
position_has_n = {}
for w in WINDOW_SIZES:
    first_seq_per_position = raw_combined_datasets[w].groupby('position_id')['Sequence'].first()
    position_has_n[w] = first_seq_per_position.str.contains('N')

valid_position_ids = set(position_has_n[FILTER_WINDOW][~position_has_n[FILTER_WINDOW]].index)

print(f"Positions valid at {FILTER_WINDOW}bp (no boundary N): {len(valid_position_ids):,} "
      f"of {len(reference_positions):,}")

print(f"\nCost of standardizing the N-filter on {FILTER_WINDOW}bp "
      f"(positions that would have survived at a smaller window, but are "
      f"dropped because their {FILTER_WINDOW}bp context hits a boundary):")

invalid_at_filter_window = set(position_has_n[FILTER_WINDOW][position_has_n[FILTER_WINDOW]].index)
dropped_would_survive = {}
for w in WINDOW_SIZES:
    if w == FILTER_WINDOW:
        continue
    would_survive_at_w = set(position_has_n[w][~position_has_n[w]].index)
    dropped = would_survive_at_w & invalid_at_filter_window
    dropped_would_survive[w] = len(dropped)
    print(f"  window={w:>3}bp: {len(dropped):,} positions dropped that would have "
          f"survived at {w}bp alone")

# Apply the single, shared filter to every window size
combined_datasets = {}
combined_paths = {}

for w in WINDOW_SIZES:
    df = raw_combined_datasets[w]
    before = len(df)
    clean_df = df[df['position_id'].isin(valid_position_ids)].reset_index(drop=True)
    after = len(clean_df)

    out_path = os.path.join(PROCESSED_DIR, f"tp53_mutation_dataset_w{w}.csv")
    clean_df.to_csv(out_path, index=False)
    combined_paths[w] = out_path
    combined_datasets[w] = clean_df

    print(f"window={w:>3}bp  total={before:>7,}  dropped={before - after:>6,}  "
          f"final={after:>7,}  -> {out_path}")

# Every window size must now describe exactly the same set of positions
final_position_sets = {w: set(combined_datasets[w]['position_id']) for w in WINDOW_SIZES}
reference_final = final_position_sets[WINDOW_SIZES[0]]
for w in WINDOW_SIZES[1:]:
    assert final_position_sets[w] == reference_final, (
        f"Post-filter position_id set for window {w} differs from window {WINDOW_SIZES[0]}."
    )
print(f"\nOK: all window sizes share the identical {len(reference_final):,} "
      "positions after the shared 101bp N-filter.")


Positions valid at 101bp (no boundary N): 13,423 of 14,355

Cost of standardizing the N-filter on 101bp (positions that would have survived at a smaller window, but are dropped because their 101bp context hits a boundary):
  window= 11bp: 855 positions dropped that would have survived at 11bp alone
  window= 21bp: 762 positions dropped that would have survived at 21bp alone
  window= 51bp: 514 positions dropped that would have survived at 51bp alone


window= 11bp  total=716,690  dropped=23,262  final=693,428  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w11.csv


window= 21bp  total=716,690  dropped=23,262  final=693,428  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w21.csv


window= 51bp  total=716,690  dropped=23,262  final=693,428  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w51.csv


window=101bp  total=716,690  dropped=23,262  final=693,428  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w101.csv



OK: all window sizes share the identical 13,423 positions after the shared 101bp N-filter.


## Dataset statistics per window size

In [5]:
for w in WINDOW_SIZES:
    df = combined_datasets[w]
    print("=" * 70)
    print(f"WINDOW SIZE: {w}bp")
    print("=" * 70)
    print(f"Total mutation instances: {len(df):,}")
    print(f"Unique genomic positions: {df['position_id'].nunique():,}")

    print("\nClass distribution:")
    class_pct = df['MutationType'].value_counts(normalize=True).sort_values(ascending=False) * 100
    for mut_type, pct in class_pct.items():
        print(f"  {mut_type}: {pct:6.2f}%")

    position_counts = df['position_id'].value_counts()
    hotspots = (position_counts >= HOTSPOT_THRESHOLD).sum()
    print(f"\nPositions with >={HOTSPOT_THRESHOLD} instances (hotspots): {hotspots}")
    print(f"Positions with <{HOTSPOT_THRESHOLD} instances (rare): {df['position_id'].nunique() - hotspots}")
    print()


WINDOW SIZE: 11bp
Total mutation instances: 693,428
Unique genomic positions: 13,423

Class distribution:
  C>T:  52.15%
  C>A:  17.14%
  T>C:  12.49%
  C>G:   7.74%
  T>A:   6.00%
  T>G:   4.48%

Positions with >=100 instances (hotspots): 1359
Positions with <100 instances (rare): 12064

WINDOW SIZE: 21bp
Total mutation instances: 693,428
Unique genomic positions: 13,423

Class distribution:
  C>T:  52.15%
  C>A:  17.14%
  T>C:  12.49%
  C>G:   7.74%
  T>A:   6.00%
  T>G:   4.48%

Positions with >=100 instances (hotspots): 1359
Positions with <100 instances (rare): 12064

WINDOW SIZE: 51bp
Total mutation instances: 693,428
Unique genomic positions: 13,423

Class distribution:
  C>T:  52.15%
  C>A:  17.14%
  T>C:  12.49%
  C>G:   7.74%
  T>A:   6.00%
  T>G:   4.48%

Positions with >=100 instances (hotspots): 1359
Positions with <100 instances (rare): 12064

WINDOW SIZE: 101bp
Total mutation instances: 693,428
Unique genomic positions: 13,423

Class distribution:
  C>T:  52.15%
  C>A:  

## Stage 5: cluster `position_id`s into locus clusters (fixes isoform-redundancy leakage)

TP53's 19 `gene_N`/`file_N` inputs are 19 transcript **isoforms**, not 19
genes -- several share long identical exonic stretches, and COSMIC
re-annotates the same genomic variant once per affected transcript. Left as
raw `position_id`s (transcript, cds_pos), this means:

- the *same* underlying mutation can appear as up to ~19 different
  `position_id`s, each getting its own slice of the total observation count
  -- fragmenting/distorting `n_instances` and therefore the hotspot/rare
  label, and
- if those `position_id`s are split into different train/test groups, the
  model sees byte-identical `Sequence` content on both sides of the split,
  even though `position_id` itself never crosses -- a leak invisible at the
  `position_id` level.

The fix: group `position_id`s that share **identical sequence at the
smallest window size (11bp)** into one `cluster_id`. This is the one
resolution that closes the leak for all four window sizes simultaneously:
every window is a substring of every larger window centered on the same
base, so identical content at any larger window implies identical content at
11bp, and 11bp is therefore the necessary (coarsest, most inclusive)
condition -- clustering on it can only merge positions that could otherwise
leak at some window size, never miss one.

Downstream, every canonical count (`n_instances`, `majority_subtype`,
`hotspot_flag`) and the train/val/test split are computed on **clusters**,
not raw `position_id`s.


In [6]:
position_to_cluster_seq = combined_datasets[CLUSTER_WINDOW].groupby('position_id')['Sequence'].first()
position_to_cluster_seq.name = 'cluster_id'  # the smallest-window sequence *is* the cluster identifier

n_positions = len(position_to_cluster_seq)
n_clusters = position_to_cluster_seq.nunique()
cluster_sizes = position_to_cluster_seq.value_counts()

print(f"{n_positions:,} valid position_ids collapse into {n_clusters:,} locus clusters "
      f"(by exact {CLUSTER_WINDOW}bp sequence identity)")
print(f"cluster size (positions sharing one cluster): "
      f"min={cluster_sizes.min()}  max={cluster_sizes.max()}  mean={cluster_sizes.mean():.1f}")
print(f"positions in a cluster of size > 1 (i.e. genuinely at risk of the leak): "
      f"{(position_to_cluster_seq.map(cluster_sizes) > 1).sum():,} / {n_positions:,}")


13,423 valid position_ids collapse into 906 locus clusters (by exact 11bp sequence identity)
cluster size (positions sharing one cluster): min=1  max=26  mean=14.8
positions in a cluster of size > 1 (i.e. genuinely at risk of the leak): 13,403 / 13,423


## Stage 6: canonical cluster-level table (computed once)

One row per `cluster_id` -- the correct resolution for "how mutated is this
locus": instance count, majority subtype, subtype diversity, and the hotspot
flag that downstream splitting is stratified on, all aggregated across every
`position_id` that shares the cluster's 11bp sequence identity.

A `position_table.csv` is still written at `position_id` granularity for QC
/ traceability (it carries the `cluster_id` each position belongs to), but it
is no longer what splitting operates on.


In [ ]:
def shannon_entropy(series):
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())


# Any window size works here since position_id, MutationType, gene_number,
# cds_pos are all window-size independent after the shared N-filter -- we
# use the largest (filter) window's data for concreteness.
basis_df = combined_datasets[FILTER_WINDOW].merge(
    position_to_cluster_seq, left_on='position_id', right_index=True
)

# --- position-level table (QC / traceability only) ---
position_table = (
    basis_df.groupby('position_id')
    .agg(
        gene_number=('gene_number', 'first'),
        cds_pos=('cds_pos', 'first'),
        cluster_id=('cluster_id', 'first'),
        n_instances=('MutationType', 'size'),
        majority_subtype=('MutationType', lambda s: s.mode()[0]),
        n_distinct_subtypes=('MutationType', 'nunique'),
        shannon_entropy=('MutationType', shannon_entropy),
    )
    .reset_index()
)

position_table_path = os.path.join(PROCESSED_DIR, 'position_table.csv')
position_table.to_csv(position_table_path, index=False)
print(f"Position-level table (QC only): {len(position_table):,} rows -> {position_table_path}")

# --- cluster-level table (this is what splitting operates on) ---
cluster_table = (
    basis_df.groupby('cluster_id')
    .agg(
        n_positions=('position_id', 'nunique'),
        n_instances=('MutationType', 'size'),
        majority_subtype=('MutationType', lambda s: s.mode()[0]),
        n_distinct_subtypes=('MutationType', 'nunique'),
        shannon_entropy=('MutationType', shannon_entropy),
    )
    .reset_index()
)
cluster_table['hotspot_flag'] = cluster_table['n_instances'] >= HOTSPOT_THRESHOLD

cluster_table_path = os.path.join(PROCESSED_DIR, 'cluster_table.csv')
cluster_table.to_csv(cluster_table_path, index=False)

print(f"Cluster-level table: {len(cluster_table):,} rows -> {cluster_table_path}")
print(f"  hotspot (n_instances >= {HOTSPOT_THRESHOLD}): {cluster_table['hotspot_flag'].sum():,}")
print(f"  rare    (n_instances <  {HOTSPOT_THRESHOLD}): {(~cluster_table['hotspot_flag']).sum():,}")
print(f"\nFor comparison, the OLD (pre-clustering, per-position_id) hotspot/rare counts were:")
old_hotspot = (position_table.groupby('position_id')['n_instances'].first() >= HOTSPOT_THRESHOLD).sum()
print(f"  hotspot: {old_hotspot:,}  rare: {len(position_table) - old_hotspot:,}")
print("(the shift illustrates how much isoform-redundant re-annotation was "
      "distorting the hotspot/rare label before clustering)")

print(f"\nMajority-subtype distribution across clusters:")
print(cluster_table['majority_subtype'].value_counts())
cluster_table.head()


## Stage 7: stratified cluster-based split -- run exactly twice (hotspot, rare)

Splitting happens once on the canonical **cluster** table, stratified by
`majority_subtype`, separately for hotspot and rare-variant clusters. This is
**not** repeated per window size: the resulting `cluster_id -> split`
assignment is the single source of truth every window size's CSVs are built
from in the next stage.


In [8]:
def stratified_position_split(cluster_df, id_col='cluster_id', strata_col='majority_subtype',
                               test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE):
    # Explicit object-dtype conversion: pandas 3.x string columns come back as
    # Arrow-backed arrays, which sklearn's fancy indexing (train_test_split's
    # internal _safe_indexing) can't slice with a numpy integer index array.
    ids = np.asarray(cluster_df[id_col].to_numpy(), dtype=object)
    strata = np.asarray(cluster_df[strata_col].to_numpy(), dtype=object)

    train_val_ids, test_ids = train_test_split(
        ids, test_size=test_size, stratify=strata, random_state=random_state
    )

    train_val_df = cluster_df[cluster_df[id_col].isin(train_val_ids)]
    train_ids, val_ids = train_test_split(
        np.asarray(train_val_df[id_col].to_numpy(), dtype=object),
        test_size=val_size / (1 - test_size),
        stratify=np.asarray(train_val_df[strata_col].to_numpy(), dtype=object),
        random_state=random_state,
    )

    train_ids, val_ids, test_ids = set(train_ids), set(val_ids), set(test_ids)

    assert not (train_ids & val_ids)
    assert not (train_ids & test_ids)
    assert not (val_ids & test_ids)

    return train_ids, val_ids, test_ids


hotspot_clusters = cluster_table[cluster_table['hotspot_flag']]
rare_clusters = cluster_table[~cluster_table['hotspot_flag']]

# No cluster may belong to both groups -- the hotspot flag is a strict partition.
assert not (set(hotspot_clusters['cluster_id']) & set(rare_clusters['cluster_id']))

hotspot_train, hotspot_val, hotspot_test = stratified_position_split(hotspot_clusters)
rare_train, rare_val, rare_test = stratified_position_split(rare_clusters)

cluster_split_assignments = {
    'hotspot': {'train': hotspot_train, 'val': hotspot_val, 'test': hotspot_test},
    'rare': {'train': rare_train, 'val': rare_val, 'test': rare_test},
}

cluster_assignment_paths = {}
for group_name, split_sets in cluster_split_assignments.items():
    rows = []
    for split_name, ids in split_sets.items():
        rows.extend({'cluster_id': cid, 'split': split_name} for cid in ids)
    assignment_df = pd.DataFrame(rows)

    path = os.path.join(SPLITS_DIR, f'cluster_assignment_{group_name}.csv')
    assignment_df.to_csv(path, index=False)
    cluster_assignment_paths[group_name] = path

    print(f"{group_name}: train={len(split_sets['train']):>4,} clusters  "
          f"val={len(split_sets['val']):>4,} clusters  test={len(split_sets['test']):>4,} clusters "
          f"-> {path}")

# Expand cluster-level split assignment down to position_id -> split, since
# that's the join key the instance-level (window-size) data uses.
position_to_cluster = position_table.set_index('position_id')['cluster_id']

split_assignments = {}
for group_name, split_sets in cluster_split_assignments.items():
    split_assignments[group_name] = {}
    for split_name, cluster_ids in split_sets.items():
        position_ids = set(position_to_cluster[position_to_cluster.isin(cluster_ids)].index)
        split_assignments[group_name][split_name] = position_ids

assignment_paths = {}
for group_name, split_sets in split_assignments.items():
    rows = []
    for split_name, ids in split_sets.items():
        rows.extend({'position_id': pid, 'split': split_name} for pid in ids)
    assignment_df = pd.DataFrame(rows)

    path = os.path.join(SPLITS_DIR, f'position_assignment_{group_name}.csv')
    assignment_df.to_csv(path, index=False)
    assignment_paths[group_name] = path

    print(f"{group_name}: train={len(split_sets['train']):>6,}  val={len(split_sets['val']):>5,}  "
          f"test={len(split_sets['test']):>5,}  positions (expanded from clusters) -> {path}")


hotspot: train= 322 clusters  val=  69 clusters  test=  70 clusters -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\cluster_assignment_hotspot.csv
rare: train= 311 clusters  val=  67 clusters  test=  67 clusters -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\cluster_assignment_rare.csv
hotspot: train= 5,304  val=1,143  test=1,136  positions (expanded from clusters) -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\position_assignment_hotspot.csv
rare: train= 4,041  val=  915  test=  884  positions (expanded from clusters) -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\position_assignment_rare.csv


## Stage 8: build per-window, per-group instance-level CSVs

For each window size, join instance rows to the position-level split
assignment expanded from the cluster split above -- nothing is re-split or
recomputed per window size, only joined.


In [9]:
window_group_split_paths = {}

for w in WINDOW_SIZES:
    df_w = combined_datasets[w]
    window_group_split_paths[w] = {}

    for group_name, split_sets in split_assignments.items():
        out_dir = os.path.join(SPLITS_DIR, f'window_{w}', group_name)
        os.makedirs(out_dir, exist_ok=True)
        window_group_split_paths[w][group_name] = {}

        for split_name, ids in split_sets.items():
            subset = (
                df_w[df_w['position_id'].isin(ids)]
                [['position_id', 'Sequence', 'MutationType']]
                .reset_index(drop=True)
            )
            path = os.path.join(out_dir, f'{split_name}.csv')
            subset.to_csv(path, index=False)
            window_group_split_paths[w][group_name][split_name] = path

    print(f"window={w:>3}bp  "
          + "  ".join(
              f"{g}/{s}={len(pd.read_csv(window_group_split_paths[w][g][s])):,}"
              for g in ('hotspot', 'rare') for s in ('train', 'val', 'test')
          ))


window= 11bp  hotspot/train=467,431  hotspot/val=97,173  hotspot/test=112,511  rare/train=11,274  rare/val=2,677  rare/test=2,362


window= 21bp  hotspot/train=467,431  hotspot/val=97,173  hotspot/test=112,511  rare/train=11,274  rare/val=2,677  rare/test=2,362


window= 51bp  hotspot/train=467,431  hotspot/val=97,173  hotspot/test=112,511  rare/train=11,274  rare/val=2,677  rare/test=2,362


window=101bp  hotspot/train=467,431  hotspot/val=97,173  hotspot/test=112,511  rare/train=11,274  rare/val=2,677  rare/test=2,362


## Verification

These checks are the actual bug fix, not just documentation of intent:
identical position sets across window sizes, no position/cluster leaking
across splits, no position shared between the hotspot and rare-variant
groups, and -- new in this revision -- **no exact-duplicate `Sequence`
content, at any window size, crosses a train/val/test boundary.**


In [10]:
print("Cross-window position-set identity check")
print("-" * 70)
for group_name in split_assignments:
    for split_name in ('train', 'val', 'test'):
        id_sets = {
            w: set(pd.read_csv(window_group_split_paths[w][group_name][split_name])['position_id'])
            for w in WINDOW_SIZES
        }
        reference = id_sets[WINDOW_SIZES[0]]
        for w in WINDOW_SIZES[1:]:
            assert id_sets[w] == reference, (
                f"{group_name}/{split_name}: position_id set at window {w} "
                f"differs from window {WINDOW_SIZES[0]}"
            )
        assert reference == split_assignments[group_name][split_name], (
            f"{group_name}/{split_name}: joined instance data doesn't match the "
            "position_id -> split assignment table"
        )
        print(f"  {group_name}/{split_name}: identical across all {len(WINDOW_SIZES)} "
              f"window sizes ({len(reference):,} positions)")

print("\nNo-overlap checks (clusters, positions, hotspot vs rare)")
print("-" * 70)
for group_name, split_sets in cluster_split_assignments.items():
    assert not (split_sets['train'] & split_sets['val'])
    assert not (split_sets['train'] & split_sets['test'])
    assert not (split_sets['val'] & split_sets['test'])
    print(f"  {group_name}: cluster train/val/test are mutually exclusive")

all_hotspot_clusters = set().union(*cluster_split_assignments['hotspot'].values())
all_rare_clusters = set().union(*cluster_split_assignments['rare'].values())
assert not (all_hotspot_clusters & all_rare_clusters)
print("  hotspot and rare-variant cluster sets are mutually exclusive")

print("\nSequence-leakage check (the actual fix): no exact-duplicate Sequence,")
print("at ANY window size, may appear in more than one of train/val/test.")
print("-" * 70)
for group_name in split_assignments:
    for w in WINDOW_SIZES:
        seq_by_split = {}
        for split_name in ('train', 'val', 'test'):
            path = window_group_split_paths[w][group_name][split_name]
            seq_by_split[split_name] = set(pd.read_csv(path)['Sequence'])

        train_val = seq_by_split['train'] & seq_by_split['val']
        train_test = seq_by_split['train'] & seq_by_split['test']
        val_test = seq_by_split['val'] & seq_by_split['test']
        assert not train_val, f"{group_name}/window={w}: Sequence overlap between train and val"
        assert not train_test, f"{group_name}/window={w}: Sequence overlap between train and test"
        assert not val_test, f"{group_name}/window={w}: Sequence overlap between val and test"
    print(f"  {group_name}: zero Sequence overlap between any pair of splits, at every window size")

print("\n101bp N-filter standardization cost (recap)")
print("-" * 70)
for w, dropped in dropped_would_survive.items():
    print(f"  window={w:>3}bp: {dropped:,} positions dropped that would have "
          f"survived using {w}bp's own N-filter")

print("\nAll verification checks passed.")


Cross-window position-set identity check
----------------------------------------------------------------------


  hotspot/train: identical across all 4 window sizes (5,304 positions)


  hotspot/val: identical across all 4 window sizes (1,143 positions)


  hotspot/test: identical across all 4 window sizes (1,136 positions)
  rare/train: identical across all 4 window sizes (4,041 positions)
  rare/val: identical across all 4 window sizes (915 positions)
  rare/test: identical across all 4 window sizes (884 positions)

No-overlap checks (clusters, positions, hotspot vs rare)
----------------------------------------------------------------------
  hotspot: cluster train/val/test are mutually exclusive
  rare: cluster train/val/test are mutually exclusive
  hotspot and rare-variant cluster sets are mutually exclusive

Sequence-leakage check (the actual fix): no exact-duplicate Sequence,
at ANY window size, may appear in more than one of train/val/test.
----------------------------------------------------------------------


  hotspot: zero Sequence overlap between any pair of splits, at every window size
  rare: zero Sequence overlap between any pair of splits, at every window size

101bp N-filter standardization cost (recap)
----------------------------------------------------------------------
  window= 11bp: 855 positions dropped that would have survived using 11bp's own N-filter
  window= 21bp: 762 positions dropped that would have survived using 21bp's own N-filter
  window= 51bp: 514 positions dropped that would have survived using 51bp's own N-filter

All verification checks passed.


## Summary

In [11]:
summary = {
    'window_sizes': WINDOW_SIZES,
    'n_filter_window': FILTER_WINDOW,
    'cluster_window': CLUSTER_WINDOW,
    'hotspot_threshold': HOTSPOT_THRESHOLD,
    'combined_datasets': combined_paths,
    'position_table': position_table_path,
    'cluster_table': cluster_table_path,
    'cluster_assignments': cluster_assignment_paths,
    'split_assignments': assignment_paths,
    'window_group_split_paths': window_group_split_paths,
    'extraction_stats': dict(extraction_stats),
    'positions_dropped_by_101bp_filter_vs_own_window': dropped_would_survive,
    'n_positions_before_clustering': int(n_positions),
    'n_clusters_after_clustering': int(n_clusters),
}

summary_path = os.path.join(PROCESSED_DIR, 'pipeline_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Pipeline summary written to {summary_path}\n")
print("Outputs:")
print(f"  position table (QC): {position_table_path}")
print(f"  cluster table (split basis): {cluster_table_path}")
for group_name, path in cluster_assignment_paths.items():
    print(f"  {group_name} cluster split assignment: {path}")
for group_name, path in assignment_paths.items():
    print(f"  {group_name} position split assignment (expanded): {path}")
for w in WINDOW_SIZES:
    print(f"  window_{w} combined dataset: {combined_paths[w]}")
    for group_name in ('hotspot', 'rare'):
        for split_name in ('train', 'val', 'test'):
            print(f"    {group_name}/{split_name}: {window_group_split_paths[w][group_name][split_name]}")


Pipeline summary written to C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\pipeline_summary.json

Outputs:
  position table (QC): C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\position_table.csv
  cluster table (split basis): C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\cluster_table.csv
  hotspot cluster split assignment: C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\cluster_assignment_hotspot.csv
  rare cluster split assignment: C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\cluster_assignment_rare.csv
  hotspot position split assignment (expanded): C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\position_assignment_hotspot.csv
  rare position split assignment (expanded): C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\position_assignment_rare.csv
  window_11 combined dataset: C:\Users\danya\Documents\projects\tp53_mutation_subt